# KYC Sanctions & Adverse Media Screening — Agentic AI

**Hult International Business School — MFIN, AI Trends in Finance — Assignment A2 (Agentic AI)**

**Domain:** Financial Compliance Monitoring → Sanctions & Adverse Media Screening (KYC)

**Business problem:** onboarding/periodic-review analysts at a bank have to manually check every new or existing
client against sanctions lists and negative news before clearing them. Doing this by hand for every client is slow
and inconsistent. This notebook builds an agent that does a first-pass triage automatically — it decides *on its own*,
client by client, how much scrutiny each case needs, and hands the analyst a short memo instead of a pile of raw
search results.

**Audience:** primarily KYC/compliance analysts and their team leads (the people who'd actually use this tool day to
day). Secondary audience: compliance execs and regulators who care about auditability of the decision logic.

**Assumptions:**
- All client names and news articles below are **synthetic/fictional** — no real client data is used, per the
  assignment brief.
- The sanctions list itself (Section C) is the **real, public OFAC SDN list** — that part *is* real-world data.
- This system is a **decision-support triage tool, not a final compliance decision**. Every output still needs a
  human compliance officer to sign off — see the Limitations cell at the end.

**Why this isn't "just RAG again":** the class notebook (`BOS_RAG_Finance_2026`) answered one fixed question over one
fixed corpus. Here, Section D adds a **router** that reads intermediate signals (sanctions score, retrieval
confidence, classifier confidence) and *branches* — different clients take different paths through the pipeline,
and one branch even re-queries the database with different search terms before deciding anything. That's the
"agentic" part the assignment is grading.


## How to read this notebook

Every section is tagged so it's obvious what's copy-pasted from class vs. what's new:

- `[REUSED]` — same code/pattern as `BOS_RAG_Finance_2026`, just pointed at a new collection/persona.
- `[NEW]` — the agentic layer built for this assignment.

| Section | What it is | Tag |
|---|---|---|
| Setup (Mongo, corpus, chunking, embeddings, vector search) | Same RAG plumbing as class | `[REUSED]` (new corpus) |
| B — Zero-shot adverse-media classifier | `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` | `[NEW]` |
| C — Sanctions list screening | OFAC SDN + cosine similarity on the *same* MiniLM embeddings | `[NEW]` |
| D — Agentic router | conditional decision logic + self-correction loop | `[NEW]` |
| E — NER confirmation | `Davlan/bert-base-multilingual-cased-ner-hrl` | `[NEW, optional]` |
| F — Memo generation | `distilgpt2`, same call as class, new persona | `[REUSED]` (new prompt) |


In [ ]:
# [REUSED FROM CLASS] same install cell as BOS_RAG_Finance_2026 - dropped "openai" since nothing here calls a
# paid API (the whole point is a $0 stack of public HF models). transformers itself isn't installed explicitly,
# same as in class - it comes along as a dependency of sentence-transformers.
!pip install -q pymupdf
!pip install -q pymongo
!pip install -q sentence-transformers
!pip install -q tiktoken


In [ ]:
# [NEW - security fix] in class the Mongo connection string was pasted straight into the notebook
# (cell-1 of BOS_RAG_Finance_2026 had `CONNECTION_STRING = "mongodb+srv://Monts2319:...`) - that means anyone who
# sees the notebook gets write access to the cluster. Here both secrets come from Colab's secret manager instead.
#
# Before running: click the key icon on the left sidebar of Colab and add:
#   MONGODB_URI  -> your Atlas connection string (mongodb+srv://...)
#   HF_TOKEN     -> a Hugging Face access token (none of the models below are gated, but it's good habit to have
#                   this wired up rather than hardcoding it later if that changes)

from google.colab import userdata

MONGODB_URI = userdata.get("MONGODB_URI")
HF_TOKEN = userdata.get("HF_TOKEN")


In [ ]:
# [REUSED FROM CLASS] identical pattern to cell-1 of BOS_RAG_Finance_2026, just a new database/collection
# for the KYC use case instead of the fraud-email one.
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)

db = client["KYC_Screening"]
collection = db["KYC_AdverseMedia"]

print("Connected. Using database:", db.name, "| collection:", collection.name)


In [ ]:
# [NEW - convenience] in class the vector index ("vector_index_2") was created by hand in the Atlas UI.
# pymongo 4.5+ can request Atlas Search/Vector Search index creation directly from the driver, so we try that here.
# If your cluster tier doesn't support the Admin API, this just prints a message - go create the index manually in
# the Atlas UI with the same field name/dimensions/similarity metric and everything downstream still works.
from pymongo.operations import SearchIndexModel

VECTOR_INDEX_NAME = "kyc_vector_index"

try:
    search_index_model = SearchIndexModel(
        definition={
            "fields": [
                {"type": "vector", "path": "embedding", "numDimensions": 384, "similarity": "cosine"}
            ]
        },
        name=VECTOR_INDEX_NAME,
        type="vectorSearch",
    )
    collection.create_search_index(model=search_index_model)
    print(f"Requested creation of '{VECTOR_INDEX_NAME}' - can take ~1 min to become queryable on Atlas.")
except Exception as e:
    print(f"Could not auto-create the index (may already exist, or needs manual setup in the Atlas UI): {e}")


## Section A — Adverse media corpus `[REUSED PATTERN, NEW CORPUS]`

Class used a folder of synthetic fraud-alert emails. Here we swap in a synthetic **adverse media corpus**: short
"news article" style write-ups for 17 fictional clients — a mix of:
- clean clients (nothing negative),
- clients whose **name** happens to resemble common sanctions-list name patterns but whose actual news coverage is
  unrelated (these exist specifically to stress-test the router against name-only false positives), and
- clients with genuine risk content spanning all four risk categories used in Section B.

We generate these **as PDFs using `fitz`** (instead of unzipping a pre-made corpus like class did) so that the
extraction cell right after it can stay exactly the same PyMuPDF read-loop from class.


In [ ]:
# [NEW] our synthetic client roster + "news article" text. Every entry becomes one PDF file.
# Ground truth (for our own sanity when demoing): profiles 1-6 = clean, 7-11 = homonym risk (name looks risky,
# content isn't), 12-17 = genuine risk (one per risk category, roughly).

CLIENT_PROFILES = [
    {"name": "Laura Bennett", "filename": "adverse_media_laura_bennett.pdf", "text":
     "Laura Bennett is the founder and CEO of Brightloom Analytics, a small SaaS startup that helps regional "
     "grocery chains forecast inventory. The company recently grew past 50 employees and opened a second office. "
     "Bennett was profiled this week for her volunteer work mentoring first-generation college students interested "
     "in data careers, and for organizing a charity coding bootcamp that raised funds for a local shelter."},

    {"name": "Carlos Medina", "filename": "adverse_media_carlos_medina.pdf", "text":
     "Carlos Medina owns Medina Family Kitchen, a small chain of Tex-Mex restaurants. He just opened his third "
     "location downtown, and local food critics gave it a strong review, praising the family recipes and the "
     "restaurant's practice of sourcing produce from nearby farms. Medina has also hosted free community dinners "
     "during the holidays for the past three years."},

    {"name": "Wei Zhang", "filename": "adverse_media_wei_zhang.pdf", "text":
     "Dr. Wei Zhang, a researcher at Meridian Genomics Lab, was awarded a research grant to continue her work on "
     "early-detection biomarkers for type 2 diabetes. Colleagues describe her as meticulous and collaborative. "
     "Her lab plans to publish preliminary results next year and is partnering with two university hospitals."},

    {"name": "Fatima Al-Sayed", "filename": "adverse_media_fatima_al_sayed.pdf", "text":
     "Fatima Al-Sayed, an independent fashion designer, launched a new sustainable clothing line made from "
     "recycled textiles. A lifestyle magazine featured her workshop this month, highlighting her partnerships with "
     "local textile recyclers and her plan to donate a portion of proceeds to environmental education programs."},

    {"name": "Tomas Novak", "filename": "adverse_media_tomas_novak.pdf", "text":
     "Tomas Novak, who runs a small accounting firm, completed a charity marathon fundraiser benefiting the "
     "regional children's hospital, raising more than expected from local sponsors. He has run the fundraiser "
     "every year for the past five years and says he plans to keep doing it as long as his knees allow."},

    {"name": "Grace Okafor", "filename": "adverse_media_grace_okafor.pdf", "text":
     "Grace Okafor opened a community bookstore that also runs free after-school literacy programs for children "
     "in the neighborhood. The store was recognized by the city council for its contribution to youth education, "
     "and Okafor says she plans to expand the literacy program to a second neighborhood next year."},

    {"name": "Viktor Petrov", "filename": "adverse_media_viktor_petrov.pdf", "text":
     "Viktor Petrov, a chess coach based in Sofia, led three of his junior students to podium finishes at a "
     "regional championship. Petrov has coached youth chess for over a decade and was interviewed about his "
     "training methods, which emphasize endgame study over rote opening memorization."},

    {"name": "Ali Hassan", "filename": "adverse_media_ali_hassan.pdf", "text":
     "Ali Hassan, owner of Hassan Bakery, is opening a second branch after strong demand for his sourdough line. "
     "The bakery also ran a fundraiser this spring donating bread to a local school's breakfast program. Hassan "
     "said he hopes to open a third location within two years."},

    {"name": "Mohammed Al-Rashid", "filename": "adverse_media_mohammed_al_rashid.pdf", "text":
     "Mohammed Al-Rashid, general manager of a mid-size hotel, oversaw a renovation project covered in a travel "
     "section feature. The article praised the hotel's new energy-efficient systems and Al-Rashid's staff training "
     "program, which was cited as a model for other properties in the region."},

    {"name": "Sergei Ivanov", "filename": "adverse_media_sergei_ivanov.pdf", "text":
     "Sergei Ivanov, a violinist, is joining the city philharmonic orchestra as a section leader starting next "
     "season. Reviewers who saw his audition performance praised his interpretation of contemporary composers. "
     "Ivanov previously taught at a regional music conservatory for six years."},

    {"name": "Youssef Khalil", "filename": "adverse_media_youssef_khalil.pdf", "text":
     "Youssef Khalil, a youth football coach, led his team to a local league championship this season. Parents "
     "and league officials credited his focus on sportsmanship as much as results. Khalil has coached the same "
     "youth club for eight years and plans to start a girls' division next season."},

    {"name": "Dmitri Orlov", "filename": "adverse_media_dmitri_orlov.pdf", "text":
     "Dmitri Orlov, who runs a commodities trading firm, is under investigation after reporters found his company "
     "allegedly routed shipments through a network of shell firms in third countries to keep exporting restricted "
     "goods to a sanctioned regime after being flagged on a watchlist. Investigators say the shell companies share "
     "directors and a registered address, and are examining whether the arrangement was designed specifically to "
     "evade sanctions enforcement. Orlov's firm has not commented publicly."},

    {"name": "Isabella Cruz", "filename": "adverse_media_isabella_cruz.pdf", "text":
     "Isabella Cruz, a former city procurement official, is facing a corruption inquiry after auditors found she "
     "allegedly accepted kickbacks from contractors in exchange for steering public infrastructure contracts their "
     "way. Internal emails reviewed by investigators reportedly show contractors referring to a standing 'processing "
     "fee' tied to contract awards during her tenure. The city's ethics board has opened a formal case."},

    {"name": "Henry Whitfield", "filename": "adverse_media_henry_whitfield.pdf", "text":
     "Henry Whitfield, an investment adviser, is under a regulatory fraud investigation after former clients "
     "alleged he ran a Ponzi-like scheme, using new investor deposits to pay out earlier investors while "
     "misrepresenting the fund's actual returns. Regulators allege Whitfield misappropriated a significant portion "
     "of client funds for personal expenses. Several investors have filed complaints seeking restitution."},

    {"name": "Nadia Kessler", "filename": "adverse_media_nadia_kessler.pdf", "text":
     "Nadia Kessler, a fund manager, is being sued in a civil lawsuit by a group of former investors alleging "
     "breach of fiduciary duty after the fund posted unexpected losses. No criminal charges have been filed and "
     "Kessler's lawyers say the losses were disclosed in line with the fund's stated risk profile. The dispute is "
     "currently proceeding through arbitration."},

    {"name": "Marcus Devereux", "filename": "adverse_media_marcus_devereux.pdf", "text":
     "Marcus Devereux, a real-estate developer, has been named in a money-laundering investigation examining "
     "whether he helped funnel proceeds from an overseas bribery scheme through a series of shell-company property "
     "purchases. Investigators allege the properties were bought at above-market prices with no clear source of "
     "funds, a pattern they say is consistent with layering illicit proceeds through real estate."},

    {"name": "Elena Vasquez", "filename": "adverse_media_elena_vasquez.pdf", "text":
     "Elena Vasquez, an executive at a state-owned utility, is accused of awarding maintenance contracts to "
     "companies secretly linked to her family members in exchange for personal payments. An anti-corruption body "
     "has opened a formal case and is reviewing years of contract records. Vasquez has denied any wrongdoing "
     "through a spokesperson."},
]

import fitz
import os

CORPUS_FOLDER = "AdverseMedia"
os.makedirs(CORPUS_FOLDER, exist_ok=True)

for profile in CLIENT_PROFILES:
    pdf = fitz.open()
    page = pdf.new_page()
    page.insert_textbox(fitz.Rect(50, 50, 545, 792), profile["text"], fontsize=11)
    pdf.save(os.path.join(CORPUS_FOLDER, profile["filename"]))
    pdf.close()

print(f"Generated {len(CLIENT_PROFILES)} synthetic adverse-media PDFs in ./{CORPUS_FOLDER}")


In [ ]:
# [REUSED FROM CLASS] identical read-loop to cell-3 of BOS_RAG_Finance_2026, just pointed at our new folder
# and no zip step (we generated the PDFs directly above instead of unzipping a pre-made corpus).
import fitz
import os

documents = []

for filename in os.listdir(CORPUS_FOLDER):
    if filename.endswith(".pdf"):
        filepath = os.path.join(CORPUS_FOLDER, filename)
        pdf = fitz.open(filepath)

        text = ""

        for page in pdf:
            text += page.get_text()

        documents.append({
            "filename": filename,
            "text": text
        })

print(f"Loaded {len(documents)} adverse-media articles")


In [ ]:
# [REUSED FROM CLASS] same fixed-size, no-overlap chunker as cell-4 of BOS_RAG_Finance_2026.

def chunk_text(text, size=500):

    chunks = []

    for i in range(0, len(text), size):
        chunks.append(text[i:i+size])

    return chunks


In [ ]:
# [REUSED FROM CLASS] same embedding loop as cell-5. `model` (all-MiniLM-L6-v2) is reused later in Section C
# for name-similarity matching too - no second embedding model gets loaded.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

records = []

for doc in documents:
    chunks = chunk_text(doc["text"])

    for chunk in chunks:
        vector = model.encode(chunk)

        records.append({
            "filename": doc["filename"],
            "text": chunk,
            "embedding": vector.tolist()
        })

print(f"Built {len(records)} chunk embeddings from {len(documents)} documents")


In [ ]:
# [REUSED FROM CLASS, +1 line] same insert_many as cell-6. Added a delete_many(sess{}) first purely so that
# re-running this cell while testing doesn't keep piling up duplicate chunks in the collection.
collection.delete_many({})
collection.insert_many(records)

print(f"Inserted {collection.count_documents({})} chunks into '{collection.name}'")


In [ ]:
# [REUSED FROM CLASS - wrapped in a function] cell-8/cell-10 of BOS_RAG_Finance_2026 did this inline for one
# fixed question. Here the router needs to run a vector search multiple times per client (and twice in a row when
# self-correcting), so it's wrapped in a function. One deliberate fix vs. class: the aggregation list is named
# `mongo_pipeline` instead of `pipeline`, because we `from transformers import pipeline` later and call it
# interleaved with this function - reusing the name `pipeline` for both would silently break one of them.

def retrieve_chunks(query_text, top_k=3):
    query_embedding = model.encode(query_text).tolist()

    mongo_pipeline = [
        {
            "$vectorSearch": {
                "index": VECTOR_INDEX_NAME,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": top_k,
            }
        },
        {
            "$project": {
                "_id": 0,
                "filename": 1,
                "text": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        },
    ]

    return list(collection.aggregate(mongo_pipeline))


In [ ]:
# quick sanity check before building anything on top of this - same spirit as cell-9/cell-11 in class
# ("verify that embedded worked" / print filenames back).
test_results = retrieve_chunks("fraud, corruption or sanctions issues involving Dmitri Orlov", top_k=3)

for r in test_results:
    print(f"{r['score']:.3f}  {r['filename']}  ->  {r['text'][:80]}...")


## Section B — Adverse-media classification `[NEW]`

**Model:** [`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-mnli-xnli)
via `pipeline("zero-shot-classification", ...)`.

Why this model: it's an NLI model fine-tuned across many languages specifically for zero-shot classification, so we
can hand it our own risk taxonomy (`Financial fraud`, `Corruption`, `Sanctions`, `Litigation`, `No risk identified`)
without training a classifier ourselves - useful in compliance, where the exact taxonomy changes by jurisdiction and
you don't want to retrain a model every time it does. It also wasn't covered in class.


In [ ]:
# [NEW] load the zero-shot classifier once, then wrap it in a small helper.
from transformers import pipeline

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
)

RISK_CATEGORIES = ["Financial fraud", "Corruption", "Sanctions", "Litigation", "No risk identified"]

def classify_chunk(text):
    result = zero_shot_classifier(text, candidate_labels=RISK_CATEGORIES)
    return result["labels"][0], result["scores"][0]

# quick test
sample_chunk = [d for d in documents if "orlov" in d["filename"]][0]["text"]
category, confidence = classify_chunk(sample_chunk)
print(f"Category: {category}  |  Confidence: {confidence:.3f}")


## Section C — Sanctions list screening `[NEW]`

This part is **structured matching, not RAG**: we pull the real public OFAC SDN list and compute a name-similarity
score for every client, reusing the exact same `all-MiniLM-L6-v2` model already loaded above (name embedding +
cosine similarity) instead of bringing in a separate fuzzy-matching library.


In [ ]:
# [NEW] download the public OFAC SDN list. The file has no header row, hence the explicit column names.
# Two known URLs are tried in order since OFAC has moved this file before; if both fail (e.g. no outbound
# access from this Colab runtime) we fall back to a small, clearly-fake illustrative sample so the rest of the
# notebook still runs end-to-end for the demo. Before any real use, always pull the live file from
# https://ofac.treasury.gov/sanctions-list-service
import pandas as pd

SDN_CSV_COLUMNS = [
    "ent_num", "SDN_Name", "SDN_Type", "SDN_Program", "Title",
    "Call_Sign", "Vess_type", "Tonnage", "GRT", "Vess_flag", "Vess_owner", "Remarks",
]

SDN_URLS = [
    "https://www.treasury.gov/ofac/downloads/sdn.csv",
    "https://sanctionslistservice.ofac.treas.gov/api/PublicationPreview/exports/SDN.CSV",
]

sdn_df = None
for url in SDN_URLS:
    try:
        sdn_df = pd.read_csv(url, header=None, names=SDN_CSV_COLUMNS, encoding="latin-1", on_bad_lines="skip")
        print(f"Loaded {len(sdn_df)} SDN entries from {url}")
        break
    except Exception as e:
        print(f"Could not load from {url}: {e}")

if sdn_df is None:
    print("Both downloads failed - falling back to a small illustrative sample list (NOT real OFAC data).")
    sdn_df = pd.DataFrame({
        "SDN_Name": [
            "PETROV, Viktor A", "AL-RASHID, Mohammed K", "IVANOV, Sergei", "ORLOV, Dmitri",
            "KHALIL, Youssef", "HASSAN, Ali M", "DEVEREUX, Marcus J", "VASQUEZ, Elena R",
        ],
        "SDN_Type": ["individual"] * 8,
    })

sdn_individuals = (
    sdn_df[sdn_df["SDN_Type"].astype(str).str.lower() == "individual"]["SDN_Name"].dropna().unique().tolist()
)

# capped for speed - CPU-only embedding of the full list (10k+ names) would be slow for a live demo.
# bump this up (or remove the cap) if you have more runtime budget.
MAX_SDN_NAMES = 3000
sdn_names = sdn_individuals[:MAX_SDN_NAMES]

print(f"Screening against {len(sdn_names)} individual SDN names")

sdn_name_embeddings = model.encode(sdn_names, convert_to_tensor=True, show_progress_bar=True)


In [ ]:
# [NEW] name-similarity score against the SDN list, reusing `model` (all-MiniLM-L6-v2) - no new library.
from sentence_transformers import util

def sanctions_score(client_name):
    query_vec = model.encode(client_name, convert_to_tensor=True)
    scores = util.cos_sim(query_vec, sdn_name_embeddings)[0]
    best_idx = int(scores.argmax())
    return float(scores[best_idx]), sdn_names[best_idx]

# quick test - one clean name, one that should score higher against common SDN name patterns
for name in ["Laura Bennett", "Dmitri Orlov"]:
    score, match = sanctions_score(name)
    print(f"{name:20s} -> score={score:.3f}  closest SDN entry: {match}")


## Section E — NER confirmation `[NEW, optional]`

**Model:** [`Davlan/bert-base-multilingual-cased-ner-hrl`](https://huggingface.co/Davlan/bert-base-multilingual-cased-ner-hrl)
via `pipeline("ner", ...)`.

Purpose: before letting a retrieved chunk influence anyone's risk tier, confirm the chunk actually **names** that
client as a person, rather than just being topically similar. This is what protects the homonym-risk clients (7-11
in our roster) from getting flagged purely because vector search returned a topically-close chunk about someone
else who shares part of their name.


In [ ]:
# [NEW]
ner_pipeline = pipeline(
    "ner",
    model="Davlan/bert-base-multilingual-cased-ner-hrl",
    aggregation_strategy="simple",
)

def ner_confirms_entity(text, client_name):
    entities = ner_pipeline(text)
    person_mentions = [e["word"] for e in entities if e["entity_group"] == "PER"]
    name_parts = [p.lower() for p in client_name.split()]
    return any(part in mention.lower() for mention in person_mentions for part in name_parts)

# quick test
print(ner_confirms_entity(sample_chunk, "Dmitri Orlov"))   # expect True
print(ner_confirms_entity(sample_chunk, "Laura Bennett"))  # expect False


## Section D — The agentic router `[NEW]` — this is the core of the assignment

This is **not** a fixed question -> retrieve -> generate pipeline. `screen_client()` below makes conditional,
multi-step decisions:

1. **High sanctions_score -> escalate immediately**, adverse media isn't even needed.
2. **Medium sanctions_score -> look at adverse media too**, and weight the decision by the zero-shot category +
   confidence.
3. **Self-correction loop:** if the first `$vectorSearch` query comes back ambiguous (low similarity across the
   top-k results), the router does **not** trust that weak result - it fires a second, more specifically-worded
   query before deciding anything. This is the visible "second look" step. Look for the `# SELF-CORRECTION LOOP`
   comment in the code below.
4. **Low sanctions_score + "No risk identified" -> clear.**
5. **Everything else -> Enhanced Due Diligence** (the catch-all/intermediate case).

The thresholds below are illustrative starting points for the demo, not calibrated production values - see the
Limitations cell at the end.


In [ ]:
# [NEW] tunable constants for the router. In a real deployment these would be calibrated against a labeled
# sample of past cases rather than picked by hand.
HIGH_SANCTIONS_THRESHOLD = 0.60   # name-match alone is strong enough to escalate on its own
MED_SANCTIONS_THRESHOLD = 0.40    # name-match is suggestive but needs adverse-media corroboration
AMBIGUITY_THRESHOLD = 0.30        # below this, the first vector search result is too weak to trust
CONFIDENCE_THRESHOLD = 0.55       # minimum zero-shot confidence to let a risky category drive an escalation


In [ ]:
# [NEW] the agentic router.
def screen_client(client_name):
    """Runs the full multi-step screening flow for one client and returns a result dict.
    Branches conditionally instead of following one fixed path - see Section D markdown above."""

    # ---- Step 1: structured sanctions-list matching (Section C) ----
    score, matched_sdn_name = sanctions_score(client_name)

    # ---- Step 2: immediate escalation branch ----
    if score >= HIGH_SANCTIONS_THRESHOLD:
        # a strong name-match against the real OFAC list is damning enough on its own - no need to even look
        # at adverse media before flagging this one for a human.
        category, confidence = "Sanctions (name-match escalation)", score
        ner_confirmed = None
        used_chunks = []
        risk_tier = "Escalate"

    else:
        # ---- Step 3: adverse media retrieval, with a visible self-correction loop ----
        query = f"Adverse media, negative news, sanctions, fraud or legal issues involving {client_name}"
        results = retrieve_chunks(query, top_k=3)

        top_score = max((r["score"] for r in results), default=0.0)

        if top_score < AMBIGUITY_THRESHOLD:
            # SELF-CORRECTION LOOP: the first query came back weak/ambiguous across the board (nothing in the
            # top-k looked confidently relevant), so instead of trusting a shaky top-1 hit, retry once with a
            # more specific, risk-flavoured query before deciding anything.
            refined_query = (
                f"{client_name} fraud corruption sanctions violation money laundering criminal investigation"
            )
            refined_results = retrieve_chunks(refined_query, top_k=3)
            if max((r["score"] for r in refined_results), default=0.0) > top_score:
                results = refined_results

        # ---- Step 4: zero-shot categorisation of the strongest retrieved chunk (Section B) ----
        top_chunk = results[0] if results else None
        if top_chunk:
            category, confidence = classify_chunk(top_chunk["text"])
        else:
            category, confidence = "No risk identified", 0.0

        # ---- Step 5: NER confirmation to rule out a homonym false positive (Section E) ----
        if top_chunk:
            ner_confirmed = ner_confirms_entity(top_chunk["text"], client_name)
            if not ner_confirmed and category != "No risk identified":
                # the retrieved chunk never actually names our client as a person - don't let someone else's
                # bad news drive this client's risk tier.
                category, confidence = "No risk identified", 0.0
        else:
            ner_confirmed = None

        used_chunks = [top_chunk] if top_chunk else []

        # ---- Step 6: fuse the sanctions_score (medium/low) with the media signal into a final tier ----
        risky_categories = {"Financial fraud", "Corruption", "Sanctions", "Litigation"}
        if score >= MED_SANCTIONS_THRESHOLD:
            # medium name-similarity alone isn't damning, but paired with a confident risky category, it is.
            if category in risky_categories and confidence >= CONFIDENCE_THRESHOLD:
                risk_tier = "Escalate"
            else:
                risk_tier = "Enhanced Due Diligence"
        else:
            if category == "No risk identified":
                risk_tier = "Clear"
            else:
                risk_tier = "Enhanced Due Diligence"

    # ---- Step 7: generate the human-readable memo (Section F) ----
    memo = generate_memo(client_name, risk_tier, used_chunks)

    return {
        "client_name": client_name,
        "sanctions_score": round(score, 3),
        "matched_sdn_name": matched_sdn_name,
        "adverse_media_category": category,
        "category_confidence": round(float(confidence), 3),
        "ner_confirmed": ner_confirmed,
        "risk_tier": risk_tier,
        "memo": memo,
    }


## Section F — Memo generation `[REUSED PATTERN, NEW PERSONA]`

Same `pipeline("text-generation", model="distilgpt2")` call as class (cell-13/14 of `BOS_RAG_Finance_2026`), just a
new persona prompt and it now cites the source filename for whatever excerpt it used.


In [ ]:
# [REUSED FROM CLASS, new persona] identical generator load + prompt/response pattern as cell-13/14 in class.
generator = pipeline("text-generation", model="distilgpt2")

def generate_memo(client_name, risk_tier, used_chunks):
    context = ""
    for c in used_chunks:
        context += f"\n---\n[Source: {c['filename']}]\n{c['text']}\n"
    if not context:
        context = "\n(no adverse media excerpt was used for this client)\n"

    hf_prompt = f"""You are an experienced KYC/compliance analyst.

Based only on the retrieved excerpts below, write a short memo justifying the risk tier assigned to the client and
cite the source filename for each excerpt used.

Client: {client_name}
Assigned risk tier: {risk_tier}

Retrieved excerpts: {context}

Memo:

"""

    hf_response = generator(
        hf_prompt,
        max_new_tokens=150,
        num_return_sequences=1,
        clean_up_tokenization_spaces=True,
    )

    return hf_response[0]["generated_text"]

# quick test on one client before running the whole roster
print(generate_memo("Dmitri Orlov", "Escalate", [retrieve_chunks("Dmitri Orlov sanctions", top_k=1)[0]])[:600])


In [ ]:
# run the router across every client in our synthetic roster.
results = []
for profile in CLIENT_PROFILES:
    print(f"Screening {profile['name']}...")
    results.append(screen_client(profile["name"]))

results_df = pd.DataFrame(results)
results_df


In [ ]:
import matplotlib.pyplot as plt

tier_order = ["Clear", "Enhanced Due Diligence", "Escalate"]
tier_counts = results_df["risk_tier"].value_counts().reindex(tier_order).fillna(0)

plt.figure(figsize=(6, 4))
plt.bar(tier_counts.index, tier_counts.values, color=["#4CAF50", "#FFC107", "#E53935"])
plt.title("Risk tier distribution across the client roster")
plt.ylabel("Number of clients")
plt.tight_layout()
plt.show()


In [ ]:
tier_colors = {"Clear": "#4CAF50", "Enhanced Due Diligence": "#FFC107", "Escalate": "#E53935"}

plt.figure(figsize=(7, 5))
for tier, group in results_df.groupby("risk_tier"):
    plt.scatter(
        group["sanctions_score"], group["category_confidence"],
        label=tier, color=tier_colors.get(tier, "gray"), s=80, edgecolor="black",
    )

for _, row in results_df.iterrows():
    plt.annotate(row["client_name"].split()[-1], (row["sanctions_score"], row["category_confidence"]),
                 fontsize=8, xytext=(4, 4), textcoords="offset points")

plt.axvline(HIGH_SANCTIONS_THRESHOLD, color="red", linestyle="--", linewidth=1, label="high sanctions threshold")
plt.axvline(MED_SANCTIONS_THRESHOLD, color="orange", linestyle="--", linewidth=1, label="medium sanctions threshold")
plt.xlabel("Sanctions name-match score")
plt.ylabel("Adverse-media category confidence")
plt.title("Sanctions score vs. adverse-media confidence, by assigned tier")
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## Limitations and risks

This system is a **decision-support triage tool**, not a compliance decision-maker. A human compliance officer must
always review and approve the final risk tier before any action is taken on a real client - this notebook only ever
produces a *recommendation*.

- **Name-matching false positives/negatives:** cosine similarity on name embeddings will flag common surnames that
  happen to overlap with SDN entries (see clients 7-11 in our roster) and can miss deliberate name variations,
  transliteration differences, or aliases that a human or a dedicated fuzzy-matching/transliteration-aware tool
  would catch. Thresholds here (`HIGH_SANCTIONS_THRESHOLD`, `MED_SANCTIONS_THRESHOLD`) were picked illustratively,
  not calibrated against a labeled dataset.
- **News staleness:** the adverse-media corpus here is static and synthetic. In production, news coverage changes
  daily - a "Clear" result today says nothing about a story that breaks tomorrow, so any real system needs
  continuous re-screening, not a one-time check.
- **Hallucination risk in the generated memo:** `distilgpt2` is a small, old decoder model with no instruction
  tuning - it can produce repetitive, incoherent, or fabricated-sounding text, as seen in the class notebook's own
  demo output. The memo should be read as a rough draft an analyst edits, never as a citable fact on its own.
- **Language-coverage bias:** the zero-shot classifier and NER model are multilingual, but both were still trained
  with far more English/high-resource-language data than low-resource languages, so accuracy will be uneven across
  markets - a risk in a KYC context that's supposed to cover global clients.
- **Small, hand-picked demo corpus:** 17 synthetic clients and ~3,000 sampled SDN names are enough to demonstrate
  the workflow, not to validate real-world precision/recall. Scaling this up would need proper evaluation against
  labeled cases.
- **Human review is mandatory:** regardless of what tier this pipeline assigns, it must be treated as a
  prioritization aid for a human KYC analyst, not an automated approve/deny system.


## Appendix — reused vs. new, at a glance

| Component | Reused from class? | New model/logic |
|---|---|---|
| PDF text extraction (`fitz`) | Yes, unchanged | — |
| Chunking (`chunk_text`, 500 chars, no overlap) | Yes, unchanged | — |
| Embeddings (`all-MiniLM-L6-v2`) | Yes, same model | Also reused for name-similarity in Section C |
| MongoDB `$vectorSearch` | Yes, same aggregation shape | New collection (`KYC_AdverseMedia`), wrapped in a function |
| Credential handling | **Fixed** | `google.colab.userdata` instead of a hardcoded connection string |
| Adverse-media classification | No | `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` (zero-shot) |
| Sanctions screening | No | OFAC SDN list + cosine similarity (structured, not RAG) |
| Agentic router | No | Conditional branching + self-correction loop (`screen_client`) |
| NER confirmation | No | `Davlan/bert-base-multilingual-cased-ner-hrl` |
| Memo generation (`distilgpt2`) | Yes, same call | New KYC-analyst persona prompt |
